# Session 3: Pydantic Validation & Tool Calling (Restaurant Kiosk)

This notebook covers structured outputs, schemas validation, and tool binding using LangChain and Pydantic. We will build a simulated restaurant kiosk that:
1. Parses conversational order text into structured data using **Pydantic**.
2. Enforces **custom validation rules** (such as business rules for discounts).
3. Executes database transactions using **SQLite** as local tools, handling potential errors gracefully.
4. Sends order confirmation messages using **Email tools**.
5. Integrates both processes using **Parallel Execution (`RunnableParallel`)**.

## Setup & Environment

Install the required dependencies and configure your environment variables.

In [ ]:
# !pip install langchain langchain-openai pydantic pandas python-dotenv


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if "OPENROUTER_API_KEY" not in os.environ:
    try:
        from google.colab import userdata
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
MODEL_NAME = os.environ.get("MODEL_NAME", "google/gemini-2.0-flash-001")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

print(f"Using OpenRouter model: {MODEL_NAME}")


## 1. Database Initialization (Local SQLite)

To make this run locally, we initialize a local SQLite file named `restaurant.db` with menu items and transactional tables.

In [ ]:
import sqlite3
import pandas as pd

DB_PATH = "restaurant.db"

with sqlite3.connect(DB_PATH) as conn:
    # 1. Create tables
    conn.execute("""
        CREATE TABLE IF NOT EXISTS menu_items (
            item_code TEXT PRIMARY KEY,
            name TEXT,
            price REAL
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS orders (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            order_type TEXT,
            payment_method TEXT,
            discount_percent REAL,
            total_amount REAL
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS order_items (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            order_id INTEGER,
            item_code TEXT,
            quantity INTEGER,
            unit_price REAL,
            line_total REAL,
            FOREIGN KEY (order_id) REFERENCES orders (id),
            FOREIGN KEY (item_code) REFERENCES menu_items (item_code)
        )
    """)
    
    # 2. Seed menu items
    cursor = conn.execute("SELECT COUNT(*) FROM menu_items")
    if cursor.fetchone()[0] == 0:
        menu_data = [
            ("paneer_tikka", "Paneer Tikka", 250.0),
            ("paneer_wrap", "Paneer Wrap", 180.0),
            ("veg_burger", "Veg Burger", 150.0),
            ("french_fries", "French Fries", 120.0),
            ("cold_coffee", "Cold Coffee", 100.0)
        ]
        conn.executemany("INSERT INTO menu_items (item_code, name, price) VALUES (?, ?, ?)", menu_data)
        conn.commit()

# Display menu items to verify setup
with sqlite3.connect(DB_PATH) as conn:
    print(pd.read_sql("SELECT * FROM menu_items", conn))

## 2. Defining Pydantic Schemas & Custom Validation Rules

We use Pydantic `BaseModel` and `Field` to define structural constraints. We can enforce **custom validation business logic** (e.g. checking field interactions) using the `@model_validator` decorator.

In [ ]:
from pydantic import BaseModel, Field, model_validator
from typing import Literal, List

class OrderItem(BaseModel):
    item_code: Literal[
        "paneer_tikka", "paneer_wrap", "veg_burger",
        "french_fries", "cold_coffee"
    ] = Field(description="The database-compatible item code name")
    quantity: int = Field(gt=0, description="The amount of items ordered")

class KioskOrder(BaseModel):
    items: List[OrderItem] = Field(description="List of items in the order")
    order_type: Literal["dine_in", "takeaway"] = Field(description="Dine in or Takeaway")
    payment_method: Literal["cash", "card", "upi"] = Field(description="The customer payment choice")
    discount_percent: int = Field(default=0, ge=0, le=100, description="Discount percentage integer, e.g. 10")
    
    # Custom validator for business logic checks
    @model_validator(mode="after")
    def validate_discount_restrictions(self) -> 'KioskOrder':
        # Business rule: Cash payments cannot receive more than a 10% discount.
        if self.payment_method == "cash" and self.discount_percent > 10:
            raise ValueError("Cash orders cannot have a discount higher than 10%!")
        return self

## 3. Extracting Structured Output from LLM

By passing our Pydantic schema to `.with_structured_output()`, we enforce structural conformity on the Gemini LLM.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model=MODEL_NAME, openai_api_key=OPENROUTER_API_KEY, openai_api_base=OPENROUTER_BASE_URL, temperature=0)
structured_llm = llm.with_structured_output(KioskOrder)

extract_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract the restaurant order details. Translate human requests to exact menu item codes. Stick strictly to the validation schema rules."),
    ("human", "{conversation}")
])

extract_chain = extract_prompt | structured_llm

conversation_text = """
Customer: Give me two paneer wraps and one cold coffee.
This is takeaway, and I'll pay using UPI.
Apply the 10 percent discount code.
"""

order = extract_chain.invoke({"conversation": conversation_text})
print(type(order))
print(order.model_dump_json(indent=2))


## 4. Binding Tools with Graceful Error Handling

We define a function as a LangChain tool by adding the `@tool` decorator. Inside the function, we catch potential database/logical exceptions so the orchestrator loop can handle failure states smoothly.

In [ ]:
from langchain_core.tools import tool

@tool(args_schema=KioskOrder)
def place_order(items: List[OrderItem], order_type: str, payment_method: str, discount_percent: int = 0):
    """Calculate and save a restaurant order in the local database."""
    try:
        with sqlite3.connect(DB_PATH) as conn:
            conn.execute("PRAGMA foreign_keys = ON")
            saved_items, subtotal = [], 0
            
            for item in items:
                row = conn.execute(
                    "SELECT price FROM menu_items WHERE item_code = ?", 
                    (item.item_code,)
                ).fetchone()
                if not row:
                    raise ValueError(f"Invalid item: {item.item_code} (not found in menu)")
                    
                price = row[0]
                line_total = price * item.quantity
                subtotal += line_total
                saved_items.append((item.item_code, item.quantity, price, line_total))
                
            total = round(subtotal * (1 - discount_percent / 100))
            
            cur = conn.execute("""
                INSERT INTO orders 
                (order_type, payment_method, discount_percent, total_amount)
                VALUES (?, ?, ?, ?)
            """, (order_type, payment_method, discount_percent, total))
            order_id = cur.lastrowid
            
            conn.executemany("""
                INSERT INTO order_items 
                (order_id, item_code, quantity, unit_price, line_total)
                VALUES (?, ?, ?, ?, ?)
            """, [(order_id, *item) for item in saved_items])
            
        return {"order_id": order_id, "status": "placed", "total_amount": total}
        
    except Exception as e:
        # Return error details back to the chain instead of crashing
        return {"status": "error", "message": str(e)}

### Executing Tool Call Loop

When a model determines a tool call should run, it emits a `tool_calls` request. We intercept this request, execute the tool securely (passing back any captured errors), and feed the response back to the model to confirm completion.

In [ ]:
from langchain_core.runnables import RunnableLambda

tools = [place_order]
tools_repo = {tool.name: tool for tool in tools}
tool_llm = llm.bind_tools(tools)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Place the kiosk order using the tool. Use valid menu item codes. After the tool succeeds, confirm the order briefly."),
    ("human", "{conversation}")
])

# Helper function to run tools dynamically in a loop, handling errors gracefully
def run_tool_loop(response):
    messages = [response]
    while response.tool_calls:
        for tool_call in response.tool_calls:
            tool = tools_repo[tool_call["name"]]
            
            # Invoke tool and append result
            tool_message = tool.invoke(tool_call)
            messages.append(tool_message)
            
            # If the tool returned an error status, we can print it out for observation
            if isinstance(tool_message.content, dict) and tool_message.content.get("status") == "error":
                print(f"[Tool Error Intercepted]: {tool_message.content.get('message')}")
            
        response = tool_llm.invoke(messages)
        messages.append(response)
    return response

order_chain = prompt | tool_llm | RunnableLambda(run_tool_loop)

kiosk_input = """
Customer: Give me two cold coffees and a portion of french fries.
I want them dine_in and will pay by card.
"""

result = order_chain.invoke({"conversation": kiosk_input})
print(f"Result: {result.content}")

# Check table to confirm transaction inserted
with sqlite3.connect(DB_PATH) as conn:
    print("\n--- Saved Orders ---")
    print(pd.read_sql("SELECT * FROM orders ORDER BY id DESC LIMIT 1", conn))

## 5. Multiple Chains & Parallel Execution

Using `RunnableParallel`, we can run multiple independent chains simultaneously (e.g., saving the order to the database and email-notifying the chef).

In [ ]:
class SendEmailInput(BaseModel):
    items: List[OrderItem]
    special_request: str | None = None

@tool(args_schema=SendEmailInput)
def send_order_email(items: List[OrderItem], special_request: str | None = None):
    """Send order details to the chef."""
    # Simulating email print output for simplicity
    item_lines = "\n".join([f"- {item.item_code}: {item.quantity}" for item in items])
    email_body = f"Items:\n{item_lines}\n\nSpecial request: {special_request or 'None'}"
    print(f"\n[EMAIL SENT TO CHEF]\n{email_body}\n")
    return {"status": "email_sent"}

tools2 = [send_order_email]
tools_repo2 = {tool.name: tool for tool in tools2}
tool_llm2 = llm.bind_tools(tools2)

email_prompt = ChatPromptTemplate.from_messages([
    ("system", "Read the order, call send_order_email to alert the chef, and confirm after it succeeds."),
    ("human", "{conversation}")
])

def run_email_tool_loop(response):
    messages = [response]
    while response.tool_calls:
        for tool_call in response.tool_calls:
            tool = tools_repo2[tool_call["name"]]
            tool_message = tool.invoke(tool_call)
            messages.append(tool_message)
        response = tool_llm2.invoke(messages)
        messages.append(response)
    return response

email_chain = email_prompt | tool_llm2 | RunnableLambda(run_email_tool_loop)

# Compose the Parallel Chain
from langchain_core.runnables import RunnableParallel

parallel_chain = RunnableParallel(
    order_db=order_chain,
    email_chef=email_chain
)

test_order = """
Give me one paneer tikka and one cold coffee.
This is takeaway, and I will pay by card.
Please make the paneer tikka less spicy!
"""

print("--- Executing Parallel Chains ---")
parallel_results = parallel_chain.invoke({"conversation": test_order})

print("\n--- Outputs ---")
print(f"Database Chain: {parallel_results['order_db'].content}")
print(f"Email Chain: {parallel_results['email_chef'].content}")